In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

volume_path = "/Volumes/workspace/insurance_raw/insurance_volume"
catalog = "workspace"
schema = "insurance_bronze"

# Create Bronze schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

# Get all CSV files from Volume
files = dbutils.fs.ls(volume_path)

for file_info in files:

    if file_info.name.endswith(".csv"):

        file_name = file_info.name

        # insurance_project_dataset.csv -> insurance_project_dataset_raw
        table_name = file_name.replace(".csv", "_raw")

        print(f"Reading: {file_name}")

        df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(file_info.path)
        )

        df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(
                f"{catalog}.{schema}.{table_name}"
            )

        print(
            f"Created: {catalog}.{schema}.{table_name}"
        )